In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path

BASE_PATH = Path("/content/drive/MyDrive/COS760_Group22_Project")

RAW_PATH = BASE_PATH / "data" / "raw"
PROCESSED_PATH = BASE_PATH / "data" / "processed"
GENERATED_PATH = BASE_PATH / "data" / "generated"
MODEL_PATH = BASE_PATH / "models"
RESULTS_PATH = BASE_PATH / "results"
REPO_PATH = BASE_PATH / "repos"

for path in [RAW_PATH, PROCESSED_PATH, GENERATED_PATH, MODEL_PATH, RESULTS_PATH, REPO_PATH]:
    path.mkdir(parents=True, exist_ok=True)

print("Project folder ready:")
print(BASE_PATH)

Project folder ready:
/content/drive/MyDrive/COS760_Group22_Project


In [ ]:
!pip install -q pandas requests tqdm datasets scikit-learn transformers accelerate evaluate openai shap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import requests
import json
import re
import random
import time

from tqdm import tqdm

In [ ]:
def clean_text_column(df):
    df["text"] = df["text"].astype(str)
    df["text"] = (
        df["text"]
        .str.replace("\n", " ", regex=False)
        .str.replace("\t", " ", regex=False)
        .str.strip()
    )

    df = df[df["text"].str.len() > 0].copy()
    df = df.drop_duplicates(subset=["text"])

    return df


def split_into_sentences(text):
    text = str(text).replace("\n", " ").strip()
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return sentences

In [ ]:
#Download and process Vuk'uzenzele


def load_vukuzenzele_sentence_level(language_code, language_name):
    #url = f"https://github.com/dsfsi/vukuzenzele-nlp/tree/master/data/huggingface/{language_code}/data.jsonl"
    url = f"https://raw.githubusercontent.com/dsfsi/vukuzenzele-nlp/master/data/huggingface/{language_code}/data.jsonl"

    response = requests.get(url)

    if response.status_code != 200:
        raise Exception(
            f"Could not download {language_code}. "
            f"Status code: {response.status_code}. URL tried: {url}"
        )

    rows = []

    for line in response.text.splitlines():
        if line.strip():
            item = json.loads(line)
            article_text = item.get("text", "")

            sentences = split_into_sentences(article_text)

            for sentence in sentences:
                rows.append({
                    "text": sentence,
                    "language": language_name,
                    "language_code": language_code,
                    "source": "vukuzenzele",
                    "label": 0
                })

    df = pd.DataFrame(rows)
    df = clean_text_column(df)

    return df

In [ ]:
#Load Xhosa, Zulu, Sepedi

xhosa_df = load_vukuzenzele_sentence_level("xho", "xhosa")
zulu_df = load_vukuzenzele_sentence_level("zul", "zulu")
sepedi_df = load_vukuzenzele_sentence_level("nso", "sepedi")

xhosa_df.to_csv(RAW_PATH / "vukuzenzele_xhosa_sentences.csv", index=False)
zulu_df.to_csv(RAW_PATH / "vukuzenzele_zulu_sentences.csv", index=False)
sepedi_df.to_csv(RAW_PATH / "vukuzenzele_sepedi_sentences.csv", index=False)

print("Xhosa:", xhosa_df.shape)
print("Zulu:", zulu_df.shape)
print("Sepedi:", sepedi_df.shape)

print("\nSamples:")
display(xhosa_df.head())
display(zulu_df.head())
display(sepedi_df.head())

Xhosa: (4167, 5)
Zulu: (4057, 5)
Sepedi: (4070, 5)

Samples:


,text,language,language_code,source,label
0,kwizikolo zikarhulumente eMzantsi Afrika banak...,xhosa,xho,vukuzenzele,0
1,Oku kwe- nzeka ngenxa yenkqubo ka- rhulumente ...,xhosa,xho,vukuzenzele,0
2,"ISebe leMfundo esiSise- ko (i-DBE), lisebenzis...",xhosa,xho,vukuzenzele,0
3,"Umlawuli we-NSNP ese- beni, uNeo Rakwena, uthi...",xhosa,xho,vukuzenzele,0
4,Uthi izikolo ezingaxha- mliyo kule nkqubo kufu...,xhosa,xho,vukuzenzele,0


,text,language,language_code,source,label
0,hulumeni eNingizimu Afrika yonkana abakwazi uk...,zulu,zul,vukuzenzele,0
1,"UMnyango Wezemfundo Eyisisekelo (i-DBE), ngoku...",zulu,zul,vukuzenzele,0
2,"Umqondisi we-NSNP kulo mnyango, uNeo Rakwena, ...",zulu,zul,vukuzenzele,0
3,Uthe izikole ezingabanda- kanywanga ohlelweni ...,zulu,zul,vukuzenzele,0
4,"Onyakeni wezimali wezi- 2015/16, isabelomali e...",zulu,zul,vukuzenzele,0


,text,language,language_code,source,label
0,dikolo tša bosetšhaba tša Afrika Borwa ba kgon...,sepedi,nso,vukuzenzele,0
1,"Kgoro ya Thuto ya Mothe- o (DBE), ka NSNP, e k...",sepedi,nso,vukuzenzele,0
2,"Molaodi wa NSNP wa kgoro, Neo Rakwena, o rile ...",sepedi,nso,vukuzenzele,0
3,O rile dikolo tšeo di ne- ng di sa akeretšwa g...,sepedi,nso,vukuzenzele,0
4,"Ngwageng wa ditšhelete wa 2015/16, tšhelete ye...",sepedi,nso,vukuzenzele,0


In [ ]:
#Download AfriSenti Swahili from GitHub

AFRISENTI_REPO = REPO_PATH / "afrisent-semeval-2023"

if not AFRISENTI_REPO.exists():
    !git clone https://github.com/afrisenti-semeval/afrisent-semeval-2023 "{AFRISENTI_REPO}"
else:
    print("AfriSenti repo already exists")

SWA_PATH = AFRISENTI_REPO / "data" / "swa"

print("Files in Swahili folder:")
for file in SWA_PATH.iterdir():
    print(file.name)

Cloning into '/content/drive/MyDrive/COS760_Group22_Project/repos/afrisent-semeval-2023'...
remote: Enumerating objects: 967, done.
remote: Counting objects: 100% (396/396), done.
remote: Compressing objects: 100% (247/247), done.
remote: Total 967 (delta 206), reused 294 (delta 139), pack-reused 571 (from 1)
Receiving objects: 100% (967/967), 32.48 MiB | 10.91 MiB/s, done.
Resolving deltas: 100% (470/470), done.
Updating files: 100% (155/155), done.
Files in Swahili folder:
dev.tsv
test.tsv
train.tsv


In [ ]:
#Process AfriSenti Swahili

swahili_parts = []

for filename in ["train.tsv", "dev.tsv", "test.tsv"]:
    file_path = SWA_PATH / filename

    df = pd.read_csv(file_path, sep="\t")

    print(filename, df.shape)
    print(df.columns.tolist())

    temp = pd.DataFrame({
        "text": df["tweet"].astype(str),
        "language": "swahili",
        "language_code": "swa",
        "source": "afrisenti",
        "label": 0
    })

    swahili_parts.append(temp)

swahili_df = pd.concat(swahili_parts, ignore_index=True)
swahili_df = clean_text_column(swahili_df)

swahili_df.to_csv(RAW_PATH / "afrisenti_swahili.csv", index=False)

print("Swahili:", swahili_df.shape)
display(swahili_df.head())

train.tsv (1810, 2)
['tweet', 'label']
dev.tsv (453, 2)
['tweet', 'label']
test.tsv (748, 2)
['tweet', 'label']
Swahili: (2999, 5)


,text,language,language_code,source,label
0,Kwani tanesco wanakataga umeme makusudinadhani...,swahili,swa,afrisenti,0
1,cjawahi kuona content yoyote zaidi ya kuwa ana...,swahili,swa,afrisenti,0
2,Bomu lililokuwa limetegwa ndani ya gari likiwa...,swahili,swa,afrisenti,0
3,Kuna video inasambaa mitandaoni jamaa amemfuma...,swahili,swa,afrisenti,0
4,Viwavijeshi wanapita katika hatua kuu 6 za uku...,swahili,swa,afrisenti,0


In [ ]:
#Combine all human datasets

human_files = [
    RAW_PATH / "vukuzenzele_xhosa_sentences.csv",
    RAW_PATH / "vukuzenzele_zulu_sentences.csv",
    RAW_PATH / "vukuzenzele_sepedi_sentences.csv",
    RAW_PATH / "afrisenti_swahili.csv"
]

human_parts = []

for file_path in human_files:
    df = pd.read_csv(file_path)
    print(file_path.name, df.shape)
    human_parts.append(df)

human_df = pd.concat(human_parts, ignore_index=True)

human_df = clean_text_column(human_df)

human_df = human_df[
    ["text", "language", "language_code", "source", "label"]
]

human_df.to_csv(
    PROCESSED_PATH / "human_text_all.csv",
    index=False
)

print("Saved human_text_all.csv")
print("Shape:", human_df.shape)

print("\nLanguages:")
print(human_df["language"].value_counts())

print("\nSources:")
print(human_df["source"].value_counts())

display(human_df.head())

vukuzenzele_xhosa_sentences.csv (4167, 5)
vukuzenzele_zulu_sentences.csv (4057, 5)
vukuzenzele_sepedi_sentences.csv (4070, 5)
afrisenti_swahili.csv (2999, 5)
Saved human_text_all.csv
Shape: (15269, 5)

Languages:
language
xhosa      4167
sepedi     4058
zulu       4045
swahili    2999
Name: count, dtype: int64

Sources:
source
vukuzenzele    12270
afrisenti       2999
Name: count, dtype: int64


,text,language,language_code,source,label
0,kwizikolo zikarhulumente eMzantsi Afrika banak...,xhosa,xho,vukuzenzele,0
1,Oku kwe- nzeka ngenxa yenkqubo ka- rhulumente ...,xhosa,xho,vukuzenzele,0
2,"ISebe leMfundo esiSise- ko (i-DBE), lisebenzis...",xhosa,xho,vukuzenzele,0
3,"Umlawuli we-NSNP ese- beni, uNeo Rakwena, uthi...",xhosa,xho,vukuzenzele,0
4,Uthi izikolo ezingaxha- mliyo kule nkqubo kufu...,xhosa,xho,vukuzenzele,0


In [ ]:
#Quality check human dataset


human_df = pd.read_csv(PROCESSED_PATH / "human_text_all.csv")

print("Shape:", human_df.shape)

print("\nColumns:")
print(human_df.columns.tolist())

print("\nMissing values:")
print(human_df.isnull().sum())

print("\nLabel distribution:")
print(human_df["label"].value_counts())

print("\nLanguage distribution:")
print(human_df["language"].value_counts())

print("\nSource distribution:")
print(human_df["source"].value_counts())

print("\nText length summary:")
human_df["word_count"] = human_df["text"].astype(str).apply(lambda x: len(x.split()))
print(human_df["word_count"].describe())

print("\nAverage words per language:")
print(human_df.groupby("language")["word_count"].mean())

display(human_df.sample(10, random_state=42))

Shape: (15269, 5)

Columns:
['text', 'language', 'language_code', 'source', 'label']

Missing values:
text             0
language         0
language_code    0
source           0
label            0
dtype: int64

Label distribution:
label
0    15269
Name: count, dtype: int64

Language distribution:
language
xhosa      4167
sepedi     4058
zulu       4045
swahili    2999
Name: count, dtype: int64

Source distribution:
source
vukuzenzele    12270
afrisenti       2999
Name: count, dtype: int64

Text length summary:
count    15269.000000
mean        22.476718
std         18.610930
min          1.000000
25%         12.000000
50%         18.000000
75%         27.000000
max        388.000000
Name: word_count, dtype: float64

Average words per language:
language
sepedi     33.305323
swahili    16.948316
xhosa      19.282217
zulu       19.002967
Name: word_count, dtype: float64


,text,language,language_code,source,label,word_count
14338,Spika wa Bunge aliposhiriki Mkutano wa Jukwaa ...,swahili,swa,afrisenti,0,18
13002,Halafu sio hata mengina huwa men hawawaachi ha...,swahili,swa,afrisenti,0,19
4955,Ukuhlukumeza ngokupha thelene nomnotho: Lokhu...,zulu,zul,vukuzenzele,0,13
8742,Phapoši ye e tla šomišwa gape ke batswetši ba...,sepedi,nso,vukuzenzele,0,17
10120,Go hwetša tshedimošo ka botlalo mabapi le go u...,sepedi,nso,vukuzenzele,0,23
9713,Go be go lapiša ebile le yunifomo e fiša kudu ...,sepedi,nso,vukuzenzele,0,41
2859,• Kufuneka utyumbe umntu oza kulawula ilifa l...,xhosa,xho,vukuzenzele,0,10
1035,Siza kupapasha umthetho omtsha kulo nyaka oza ...,xhosa,xho,vukuzenzele,0,35
9243,"Kgwedi yeo e tlago e tla be e le Paseka, e leg...",sepedi,nso,vukuzenzele,0,36
8271,Disenthara di phethaga-ditšwe ke Lekgotla la B...,sepedi,nso,vukuzenzele,0,32


In [ ]:
#Save cleaned human dataset

human_df = human_df.drop(columns=["word_count"], errors="ignore")

human_df.to_csv(
    PROCESSED_PATH / "human_text_clean.csv",
    index=False
)

print("Saved human_text_clean.csv")
print(human_df.shape)

Saved human_text_clean.csv
(15269, 5)


In [ ]:
#Create seed samples for GPT generation

human_df = pd.read_csv(PROCESSED_PATH / "human_text_clean.csv")

SEED_SAMPLES_PER_LANGUAGE = 1000

seed_parts = []

for language in ["xhosa", "zulu", "sepedi", "swahili"]:
    lang_df = human_df[human_df["language"] == language].copy()

    sample_size = min(SEED_SAMPLES_PER_LANGUAGE, len(lang_df))

    sampled = lang_df.sample(
        n=sample_size,
        random_state=42
    )

    seed_parts.append(sampled)

seed_df = pd.concat(seed_parts, ignore_index=True)

seed_df.to_csv(
    PROCESSED_PATH / "gpt_seed_human_text.csv",
    index=False
)

print("Saved gpt_seed_human_text.csv")
print(seed_df.shape)
print(seed_df["language"].value_counts())

display(seed_df.head())

Saved gpt_seed_human_text.csv
(4000, 5)
language
xhosa      1000
zulu       1000
sepedi     1000
swahili    1000
Name: count, dtype: int64


,text,language,language_code,source,label
0,Saqalisa ngeSi bonelelo esiKhethekileyo se-COV...,xhosa,xho,vukuzenzele,0
1,Uxwebhu olunophawu lwesta mpu sesizwe kuthetha...,xhosa,xho,vukuzenzele,0
2,"""Okuyinene kukuba sonke isixeko saseGeorge sic...",xhosa,xho,vukuzenzele,0
3,“Sizimisele ukwenza konke okusemandleni ethu u...,xhosa,xho,vukuzenzele,0
4,Ukuqeshwa kweBhunga elitsha le-B-BBEE kuza ku ...,xhosa,xho,vukuzenzele,0


In [ ]:
# Install provider SDKs safely for Colab


!pip install -q --upgrade openai groq tqdm
!pip install -q "google-genai>=1.66.0,<2.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 11.6 MB/s eta 0:00:00


In [ ]:
import google.genai
print("google-genai loaded successfully")

google-genai loaded successfully


In [ ]:
#Add API keys


import os
import getpass

#os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")
os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API key: ")
os.environ["GEMINI_API_KEY"] = getpass.getpass("Gemini API key: ")

Groq API key: ··········
Gemini API key: ··········


In [ ]:
#Load clients and seed data

import pandas as pd
import random
import time
import csv
from pathlib import Path
from tqdm import tqdm

from openai import OpenAI
from groq import Groq
from google import genai
from google.genai import types

#openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])
gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

seed_df = pd.read_csv(PROCESSED_PATH / "gpt_seed_human_text.csv")

OUTPUT_FILE = GENERATED_PATH / "machine_text_multi_provider_raw.csv"
FAILED_FILE = GENERATED_PATH / "machine_text_generation_failures.csv"

print(seed_df.shape)
print(seed_df["language"].value_counts())

(4000, 5)
language
xhosa      1000
zulu       1000
sepedi     1000
swahili    1000
Name: count, dtype: int64


In [ ]:
#Generator configuration

providers = [

    {
        "provider": "groq",
        "models": ["llama-3.3-70b-versatile"],
        "temperatures": [0.7, 0.9, 1.1]
    },
    {
        "provider": "gemini",
        "models": ["gemini-2.5-flash"],
        "temperatures": [0.7, 0.9, 1.1]
    }
]

languages = {
    "xhosa": "xho",
    "zulu": "zul",
    "sepedi": "nso",
    "swahili": "swa"
}

task_types = [
    "question",
    "complaint",
    "request",
    "opinion",
    "casual_thought",
    "academic_query",
    "shopping",
    "technology",
    "work",
    "relationships",
    "everyday_life",
    "rewrite",
    "summarize",
    "explain",
    "educational",
    "government_notice",
    "community_information",
    "health_advice",
    "public_service_message",
    "short_story"
]

domains = [
    "weather and nature",
    "food and cooking",
    "family and relationships",
    "health and medicine",
    "education and schools",
    "government policies",
    "public services",
    "social grants",
    "community development",
    "elections and democracy",
    "shopping and prices",
    "phones and technology",
    "workplace issues",
    "transport",
    "housing",
    "school applications",
    "clinic visits",
    "water and electricity",
    "youth unemployment",
    "small businesses"
]

length_styles = [
    "one short sentence",
    "two natural sentences",
    "a short paragraph",
    "a longer paragraph",
    "a message someone might send on a phone",
    "a formal paragraph",
    "an informal paragraph"
]

In [ ]:
#Helper functions: safe saving and resume
# UPDATED: uniqueness includes task/domain/style


def append_row_csv(file_path, row, fieldnames):
    file_exists = Path(file_path).exists()

    with open(file_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)

        if not file_exists:
            writer.writeheader()

        writer.writerow(row)
        f.flush()


output_fields = [
    "text",
    "language",
    "language_code",
    "source",
    "model",
    "temperature",
    "generation_task",
    "domain",
    "length_style",
    "seed_id",
    "label"
]

failure_fields = [
    "provider",
    "model",
    "language",
    "seed_id",
    "generation_task",
    "domain",
    "length_style",
    "error"
]


def load_completed_keys():
    if not OUTPUT_FILE.exists():
        return set()

    done_df = pd.read_csv(OUTPUT_FILE)

    if done_df.empty:
        return set()

    required_cols = [
        "source",
        "model",
        "language",
        "seed_id",
        "generation_task",
        "domain",
        "length_style"
    ]

    for col in required_cols:
        if col not in done_df.columns:
            done_df[col] = ""

    return set(
        zip(
            done_df["source"].astype(str),
            done_df["model"].astype(str),
            done_df["language"].astype(str),
            done_df["seed_id"].astype(str),
            done_df["generation_task"].astype(str),
            done_df["domain"].astype(str),
            done_df["length_style"].astype(str)
        )
    )


def clean_generated_text(text):
    text = str(text).strip()
    text = text.replace("\n", " ").replace("\t", " ")
    text = " ".join(text.split())

    banned_phrases = [
        "as an ai",
        "language model",
        "dataset",
        "classifier",
        "generated text",
        "machine-generated",
        "chatgpt",
        "openai",
        "gemini",
        "groq"
    ]

    lower = text.lower()

    for phrase in banned_phrases:
        if phrase in lower:
            return None

    if len(text) == 0:
        return None

    return text

print("Updated checkpointing helpers ready")

Updated checkpointing helpers ready


In [ ]:
#Prompt builder

def build_prompt(language, task, domain, length_style, seed_text):
    prompt_styles = [
        f"""
Write naturally in {language}.

Situation: {task}
Topic: {domain}
Style: {length_style}

Use the reference only as background context. Do not copy it.

Reference:
{seed_text}

Return only the final text.
""",
        f"""
Create a realistic piece of everyday writing in {language}.

It should sound like something a real person could write.
Context: {task}
General topic: {domain}
Length/style: {length_style}

Use the background only for ideas, not wording.

Background:
{seed_text}

Return only the text.
""",
        f"""
Write in {language} only.

Produce a natural response for this scenario:
- Scenario type: {task}
- Topic area: {domain}
- Writing shape: {length_style}

The reference is only inspiration. Avoid repeating its wording.

Reference:
{seed_text}

Return only the result.
""",
        f"""
In {language}, write something realistic and natural.

The writing should fit this use case: {task}.
The topic should relate to: {domain}.
The form should be: {length_style}.

Do not copy phrases from the reference.

Reference:
{seed_text}

Return only the written text.
"""
    ]

    return random.choice(prompt_styles)

print("Prompt builder ready")

Prompt builder ready


In [ ]:
#Provider call functions
'''
def call_openai_model(model, prompt, temperature):
    response = openai_client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "Write natural text only. Do not mention AI, models, datasets, "
                    "classifiers, generation, or this instruction."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=temperature,
        max_tokens=260
    )

    return response.choices[0].message.content

'''
def call_groq_model(model, prompt, temperature):
    response = groq_client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "Write natural text only. Do not mention AI, models, datasets, "
                    "classifiers, generation, or this instruction."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=temperature,
        max_tokens=260
    )

    return response.choices[0].message.content


def call_gemini_model(model, prompt, temperature):
    response = gemini_client.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=temperature,
            max_output_tokens=260,
            system_instruction=(
                "Write natural text only. Do not mention AI, models, datasets, "
                "classifiers, generation, or this instruction."
            )
        )
    )

    return response.text

print("Provider call functions ready")

Provider call functions ready


In [ ]:
# Adaptive multi-provider generation
# UPDATED: allows multiple diverse generations per seed


MAX_TOTAL_GENERATIONS = 10000
MAX_CONSECUTIVE_PROVIDER_ERRORS = 5

completed_keys = load_completed_keys()

print("Already completed unique combinations:", len(completed_keys))

seed_df = seed_df.reset_index(drop=True)
seed_df["seed_id"] = seed_df.index.astype(str)

provider_status = {}

for p in providers:
    provider_status[p["provider"]] = {
        "active": True,
        "errors": 0
    }

total_generated = 0
attempts = 0
max_attempts = MAX_TOTAL_GENERATIONS * 20

while total_generated < MAX_TOTAL_GENERATIONS and attempts < max_attempts:

    attempts += 1

    active_providers = [
        p for p in providers
        if provider_status[p["provider"]]["active"]
    ]

    if len(active_providers) == 0:
        print("\nAll providers exhausted or rate-limited.")
        break

    provider_info = random.choice(active_providers)

    provider = provider_info["provider"]
    model = random.choice(provider_info["models"])
    temperature = random.choice(provider_info["temperatures"])

    row = seed_df.sample(n=1).iloc[0]

    language = row["language"]
    language_code = row["language_code"]
    seed_id = str(row["seed_id"])

    task = random.choice(task_types)
    domain = random.choice(domains)
    length_style = random.choice(length_styles)

    key = (
        provider,
        model,
        language,
        seed_id,
        task,
        domain,
        length_style
    )

    if key in completed_keys:
        continue

    prompt = build_prompt(
        language=language,
        task=task,
        domain=domain,
        length_style=length_style,
        seed_text=str(row["text"])[:1000]
    )

    try:

        if provider == "groq":
            raw_text = call_groq_model(
                model,
                prompt,
                temperature
            )

        elif provider == "gemini":
            raw_text = call_gemini_model(
                model,
                prompt,
                temperature
            )

        else:
            raise ValueError(
                f"Unknown provider {provider}"
            )

        cleaned_text = clean_generated_text(raw_text)

        if cleaned_text is None:
            raise ValueError(
                "Generated text failed cleaning"
            )

        output_row = {
            "text": cleaned_text,
            "language": language,
            "language_code": language_code,
            "source": provider,
            "model": model,
            "temperature": temperature,
            "generation_task": task,
            "domain": domain,
            "length_style": length_style,
            "seed_id": seed_id,
            "label": 1
        }

        append_row_csv(
            OUTPUT_FILE,
            output_row,
            output_fields
        )

        completed_keys.add(key)

        provider_status[provider]["errors"] = 0

        total_generated += 1

        if total_generated % 25 == 0:
            print(f"\nGenerated in this run: {total_generated}")

            current_df = pd.read_csv(OUTPUT_FILE)
            print("Total saved rows:", len(current_df))
            print(current_df["source"].value_counts().to_dict())
            print(current_df["language"].value_counts().to_dict())

        time.sleep(
            random.uniform(0.4, 1.5)
        )

    except Exception as e:

        error_message = str(e)

        append_row_csv(
            FAILED_FILE,
            {
                "provider": provider,
                "model": model,
                "language": language,
                "seed_id": seed_id,
                "generation_task": task,
                "domain": domain,
                "length_style": length_style,
                "error": error_message[:500]
            },
            failure_fields
        )

        print(
            f"\n[{provider}] error:",
            error_message[:200]
        )

        lower_error = error_message.lower()

        temporary_keywords = [
            "503",
            "unavailable",
            "high demand",
            "overloaded",
            "timeout"
        ]

        rate_limit_keywords = [
            "quota",
            "limit",
            "rate",
            "429",
            "too many requests",
            "resource exhausted"
        ]

        if any(k in lower_error for k in temporary_keywords):

            print(f"{provider} temporarily overloaded. Waiting...")
            time.sleep(20)

        elif any(k in lower_error for k in rate_limit_keywords):

            provider_status[provider]["errors"] += 1

            print(f"{provider} appears rate limited.")

            if (
                provider_status[provider]["errors"]
                >= MAX_CONSECUTIVE_PROVIDER_ERRORS
            ):
                provider_status[provider]["active"] = False
                print(f"{provider} disabled.")

            time.sleep(15)

        else:

            provider_status[provider]["errors"] += 1

            if (
                provider_status[provider]["errors"]
                >= MAX_CONSECUTIVE_PROVIDER_ERRORS
            ):
                provider_status[provider]["active"] = False
                print(f"{provider} disabled after repeated errors.")

            time.sleep(3)

print("\nFinished safely.")

print("\nProvider status:")
print(provider_status)

print("\nSaved file:")
print(OUTPUT_FILE)

Already completed unique combinations: 2261

[gemini] error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.
gemini appears rate limited.

[gemini] error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.
gemini appears rate limited.

Generated in this run: 25
Total saved rows: 2286
{'groq': 2183, 'gemini': 103}
{'xhosa': 611, 'sepedi': 573, 'zulu': 567, 'swahili': 535}

[gemini] error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.
gemini appears rate limited.

[gemini] error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 

In [ ]:
#Inspect generated machine data


machine_df = pd.read_csv(OUTPUT_FILE)

print("Shape:", machine_df.shape)

print("\nProvider distribution:")
print(machine_df["source"].value_counts())

print("\nLanguage distribution:")
print(machine_df["language"].value_counts())

print("\nGeneration task distribution:")
print(machine_df["generation_task"].value_counts())

print("\nLength style distribution:")
print(machine_df["length_style"].value_counts())

print("\nAverage word count per provider:")
machine_df["word_count"] = (
    machine_df["text"]
    .astype(str)
    .apply(lambda x: len(x.split()))
)

print(
    machine_df.groupby("source")["word_count"]
    .mean()
)

print("\nAverage word count by language:")
print(
    machine_df.groupby("language")["word_count"]
    .mean()
)

display(
    machine_df.sample(20, random_state=42)
)

Shape: (2575, 11)

Provider distribution:
source
groq      2464
gemini     111
Name: count, dtype: int64

Language distribution:
language
xhosa      690
sepedi     647
zulu       636
swahili    602
Name: count, dtype: int64

Generation task distribution:
generation_task
health_advice             160
opinion                   151
request                   144
government_notice         142
educational               141
everyday_life             131
technology                131
rewrite                   131
short_story               130
explain                   130
work                      129
summarize                 122
shopping                  122
public_service_message    121
complaint                 121
academic_query            119
casual_thought            118
question                  112
relationships             112
community_information     108
Name: count, dtype: int64

Length style distribution:
length_style
two natural sentences                      386
a formal paragr

,text,language,language_code,source,model,temperature,generation_task,domain,length_style,seed_id,label,word_count
1644,Ndithi ke ngamalungi selelo okugqibezela izint...,xhosa,xho,groq,llama-3.3-70b-versatile,0.9,opinion,government policies,an informal paragraph,741,1,52
1189,Izitikki zethu ziklanyelwa nguqulo lwezinkonzo...,zulu,zul,groq,llama-3.3-70b-versatile,0.9,technology,shopping and prices,one short sentence,1302,1,6
495,Nimefurahishwa sana kuona biashara ndogo ndogo...,swahili,swa,groq,llama-3.3-70b-versatile,1.1,relationships,small businesses,a short paragraph,3787,1,73
1656,Tshintshi zenuza ithemba lokunyisa izithulo.,xhosa,xho,groq,llama-3.3-70b-versatile,1.1,community_information,shopping and prices,one short sentence,771,1,5
651,Imilando iyaphakama ngokuthi abantu bangenasif...,xhosa,xho,groq,llama-3.3-70b-versatile,0.7,summarize,elections and democracy,two natural sentences,602,1,27
765,Imibhikisho iyinhlangano enabantu kanye nemiga...,zulu,zul,groq,llama-3.3-70b-versatile,0.9,explain,elections and democracy,a formal paragraph,1897,1,64
254,Abantwana bethu bahlula ngokuthemba lweenkqubo...,xhosa,xho,groq,llama-3.3-70b-versatile,0.9,educational,youth unemployment,a short paragraph,543,1,24
1681,Kudala ukhona ukuthetha ngendlela ezimnandi zo...,xhosa,xho,groq,llama-3.3-70b-versatile,0.7,technology,transport,an informal paragraph,994,1,41
2187,"Kuleli thuba elihle kakhulu ezweni lethu, laph...",zulu,zul,groq,llama-3.3-70b-versatile,1.1,everyday_life,water and electricity,a longer paragraph,1952,1,70
2572,Ndifuna ukwazi ukuba ngowaphi na ukuthetha ngo...,xhosa,xho,groq,llama-3.3-70b-versatile,0.9,shopping,government policies,a longer paragraph,68,1,62


In [ ]:
#Quality audit

bad_rows = []

for idx, row in machine_df.iterrows():

    text = str(row["text"]).strip()

    problems = []

    if len(text) == 0:
        problems.append("empty")

    if len(text.split()) < 3:
        problems.append("too_short")

    banned_words = [
        "chatgpt",
        "openai",
        "gemini",
        "language model",
        "as an ai",
        "dataset",
        "classifier"
    ]

    lower_text = text.lower()

    for word in banned_words:
        if word in lower_text:
            problems.append("banned_phrase")
            break

    if problems:
        bad_rows.append({
            "index": idx,
            "text": text[:200],
            "problems": problems
        })

bad_df = pd.DataFrame(bad_rows)

print("Potential issues:", len(bad_df))

display(bad_df.head(20))

Potential issues: 2


,index,text,problems
0,5,Ukhetho luwumgog,[too_short]
1,27,Iziphumo zemigaqo-nk,[too_short]


In [ ]:
#Compare human vs machine lengths


human_df = pd.read_csv(
    PROCESSED_PATH / "human_text_clean.csv"
)

machine_df = pd.read_csv(OUTPUT_FILE)

human_df["word_count"] = (
    human_df["text"]
    .astype(str)
    .apply(lambda x: len(x.split()))
)

machine_df["word_count"] = (
    machine_df["text"]
    .astype(str)
    .apply(lambda x: len(x.split()))
)

print("Human average:")
print(human_df["word_count"].describe())

print("\nMachine average:")
print(machine_df["word_count"].describe())

print("\nHuman by language:")
print(
    human_df.groupby("language")["word_count"]
    .mean()
)

print("\nMachine by language:")
print(
    machine_df.groupby("language")["word_count"]
    .mean()
)

Human average:
count    15269.000000
mean        22.476718
std         18.610930
min          1.000000
25%         12.000000
50%         18.000000
75%         27.000000
max        388.000000
Name: word_count, dtype: float64

Machine average:
count    2575.000000
mean       44.094757
std        31.848874
min         2.000000
25%        19.000000
50%        37.000000
75%        63.000000
max       157.000000
Name: word_count, dtype: float64

Human by language:
language
sepedi     33.305323
swahili    16.948316
xhosa      19.282217
zulu       19.002967
Name: word_count, dtype: float64

Machine by language:
language
sepedi     59.828439
swahili    49.234219
xhosa      34.401449
zulu       33.740566
Name: word_count, dtype: float64


In [ ]:
#Clean machine dataset properly


human_df = pd.read_csv(PROCESSED_PATH / "human_text_clean.csv")
machine_df = pd.read_csv(OUTPUT_FILE)

print("Original machine shape:", machine_df.shape)

# -------------------------------
# Word counts
# -------------------------------
human_df["word_count"] = human_df["text"].astype(str).apply(lambda x: len(x.split()))
machine_df["word_count"] = machine_df["text"].astype(str).apply(lambda x: len(x.split()))

# -------------------------------
# Remove duplicate machine texts
# -------------------------------
machine_df = machine_df.drop_duplicates(subset=["text"]).copy()

# -------------------------------
# Remove banned phrases
# -------------------------------
bad_patterns = [
    "as an ai",
    "language model",
    "chatgpt",
    "openai",
    "gemini",
    "groq",
    "dataset",
    "classifier",
    "machine-generated",
    "generated text"
]

pattern = "|".join(bad_patterns)

machine_df = machine_df[
    ~machine_df["text"].str.lower().str.contains(pattern, na=False)
].copy()

# -------------------------------
# Remove very short junk
# General rule: at least 4 words
# Gemini stricter because its average was ~4 words
# -------------------------------
machine_df = machine_df[
    machine_df["word_count"] >= 4
].copy()

machine_df = machine_df[
    ~(
        (machine_df["source"] == "gemini") &
        (machine_df["word_count"] < 8)
    )
].copy()

# -------------------------------
# Language-specific length filtering
# Keep machine rows within a realistic range
# based on human sentence lengths per language
# -------------------------------
filtered_parts = []

for language in machine_df["language"].unique():
    human_lang = human_df[human_df["language"] == language]
    machine_lang = machine_df[machine_df["language"] == language]

    lower = max(3, int(human_lang["word_count"].quantile(0.05)))
    upper = int(human_lang["word_count"].quantile(0.95))

    # Allow some extra length, but not too much
    upper = min(max(upper + 15, 25), 90)

    filtered_lang = machine_lang[
        (machine_lang["word_count"] >= lower) &
        (machine_lang["word_count"] <= upper)
    ].copy()

    print(language)
    print("Human 5%-95%:", lower, upper)
    print("Before:", len(machine_lang), "After:", len(filtered_lang))

    filtered_parts.append(filtered_lang)

machine_clean = pd.concat(filtered_parts, ignore_index=True)

# -------------------------------
# Save cleaned machine data
# -------------------------------
machine_clean.to_csv(
    PROCESSED_PATH / "machine_text_clean.csv",
    index=False
)

print("\nFinal cleaned machine shape:", machine_clean.shape)

print("\nLanguages:")
print(machine_clean["language"].value_counts())

print("\nProviders:")
print(machine_clean["source"].value_counts())

print("\nWord count summary:")
print(machine_clean["word_count"].describe())

Original machine shape: (2575, 11)
xhosa
Human 5%-95%: 6 56
Before: 663 After: 517
sepedi
Human 5%-95%: 10 87
Before: 627 After: 454
zulu
Human 5%-95%: 6 56
Before: 607 After: 462
swahili
Human 5%-95%: 5 49
Before: 565 After: 307

Final cleaned machine shape: (1740, 12)

Languages:
language
xhosa      517
zulu       462
sepedi     454
swahili    307
Name: count, dtype: int64

Providers:
source
groq    1740
Name: count, dtype: int64

Word count summary:
count    1740.000000
mean       31.004598
std        16.949066
min         5.000000
25%        18.000000
50%        29.000000
75%        42.000000
max        87.000000
Name: word_count, dtype: float64


In [ ]:
#Build final realistic dataset


human_df = pd.read_csv(
    PROCESSED_PATH / "human_text_clean.csv"
)

machine_df = pd.read_csv(
    PROCESSED_PATH / "machine_text_clean.csv"
)

# ------------------------------------------
# Sample human per language
# Target ≈ 1.5x machine
# ------------------------------------------

human_parts = []

for language in machine_df["language"].unique():

    machine_lang = machine_df[
        machine_df["language"] == language
    ]

    human_lang = human_df[
        human_df["language"] == language
    ]

    target_human = min(
        len(human_lang),
        int(len(machine_lang) * 1.5)
    )

    sampled_human = human_lang.sample(
        n=target_human,
        random_state=42
    )

    human_parts.append(sampled_human)

balanced_human = pd.concat(
    human_parts,
    ignore_index=True
)

# ------------------------------------------
# Combine
# ------------------------------------------

final_df = pd.concat(
    [balanced_human, machine_df],
    ignore_index=True
)

# Shuffle
final_df = final_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# Save
final_df.to_csv(
    PROCESSED_PATH /
    "final_dataset.csv",
    index=False
)

print("Final dataset shape:", final_df.shape)

print("\nLabels:")
print(final_df["label"].value_counts())

print("\nLanguages:")
print(final_df["language"].value_counts())

print("\nSources:")
print(final_df["source"].value_counts())

Final dataset shape: (4349, 12)

Labels:
label
0    2609
1    1740
Name: count, dtype: int64

Languages:
language
xhosa      1292
zulu       1155
sepedi     1135
swahili     767
Name: count, dtype: int64

Sources:
source
vukuzenzele    2149
groq           1740
afrisenti       460
Name: count, dtype: int64


In [ ]:
# Create 80/10/10 split

from sklearn.model_selection import train_test_split

final_df = pd.read_csv(
    PROCESSED_PATH / "final_dataset.csv"
)

print("Original shape:", final_df.shape)

# ------------------------------------------
# First split:
# 80 train
# 20 temp
# ------------------------------------------

train_df, temp_df = train_test_split(
    final_df,
    test_size=0.20,
    random_state=42,
    stratify=final_df["label"]
)

# ------------------------------------------
# Split temp into:
# 10 validation
# 10 test
# ------------------------------------------

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

# ------------------------------------------
# Save
# ------------------------------------------

train_df.to_csv(
    PROCESSED_PATH / "train.csv",
    index=False
)

val_df.to_csv(
    PROCESSED_PATH / "validation.csv",
    index=False
)

test_df.to_csv(
    PROCESSED_PATH / "test.csv",
    index=False
)

print("\nTrain shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain labels:")
print(train_df["label"].value_counts())

print("\nValidation labels:")
print(val_df["label"].value_counts())

print("\nTest labels:")
print(test_df["label"].value_counts())

Original shape: (4349, 12)

Train shape: (3479, 12)
Validation shape: (435, 12)
Test shape: (435, 12)

Train labels:
label
0    2087
1    1392
Name: count, dtype: int64

Validation labels:
label
0    261
1    174
Name: count, dtype: int64

Test labels:
label
0    261
1    174
Name: count, dtype: int64


In [ ]:
#Train baseline model
# TF-IDF + Logistic Regression


import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

train_df = pd.read_csv(PROCESSED_PATH / "train.csv")
val_df = pd.read_csv(PROCESSED_PATH / "validation.csv")
test_df = pd.read_csv(PROCESSED_PATH / "test.csv")

X_train = train_df["text"].astype(str)
y_train = train_df["label"]

X_val = val_df["text"].astype(str)
y_val = val_df["label"]

X_test = test_df["text"].astype(str)
y_test = test_df["label"]

baseline_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),
        lowercase=True
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

baseline_model.fit(X_train, y_train)

print("Baseline model trained")

Baseline model trained


In [ ]:
#Baseline validation evaluation

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

val_pred = baseline_model.predict(X_val)

print("Validation Accuracy:", accuracy_score(y_val, val_pred))
print("Validation Macro F1:", f1_score(y_val, val_pred, average="macro"))

print("\nValidation classification report:")
print(classification_report(y_val, val_pred))

print("\nValidation confusion matrix:")
print(confusion_matrix(y_val, val_pred))

Validation Accuracy: 0.9126436781609195
Validation Macro F1: 0.909664232938399

Validation classification report:
              precision    recall  f1-score   support

           0       0.94      0.91      0.93       261
           1       0.87      0.91      0.89       174

    accuracy                           0.91       435
   macro avg       0.91      0.91      0.91       435
weighted avg       0.91      0.91      0.91       435


Validation confusion matrix:
[[238  23]
 [ 15 159]]


In [ ]:

# Baseline final test evaluation

test_pred = baseline_model.predict(X_test)

baseline_test_accuracy = accuracy_score(y_test, test_pred)
baseline_test_macro_f1 = f1_score(y_test, test_pred, average="macro")

print("Baseline Test Accuracy:", baseline_test_accuracy)
print("Baseline Test Macro F1:", baseline_test_macro_f1)

print("\nTest classification report:")
print(classification_report(y_test, test_pred))

print("\nTest confusion matrix:")
print(confusion_matrix(y_test, test_pred))

Baseline Test Accuracy: 0.9195402298850575
Baseline Test Macro F1: 0.916267482084816

Test classification report:
              precision    recall  f1-score   support

           0       0.93      0.93      0.93       261
           1       0.90      0.90      0.90       174

    accuracy                           0.92       435
   macro avg       0.92      0.92      0.92       435
weighted avg       0.92      0.92      0.92       435


Test confusion matrix:
[[243  18]
 [ 17 157]]


In [ ]:
#Save baseline predictions


baseline_results_df = test_df.copy()
baseline_results_df["baseline_prediction"] = test_pred
baseline_results_df["correct"] = (
    baseline_results_df["label"] ==
    baseline_results_df["baseline_prediction"]
)

baseline_results_df.to_csv(
    RESULTS_PATH / "baseline_test_predictions.csv",
    index=False
)

print("Saved baseline predictions")

Saved baseline predictions


In [ ]:
# Install AfroXLMR libraries


!pip install -q transformers datasets evaluate accelerate sentencepiece

In [ ]:
#Imports


import numpy as np
import pandas as pd
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    f1_score
)

print("Torch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

Torch version: 2.10.0+cu128
GPU available: True


In [ ]:
#Load split datasets

train_df = pd.read_csv(
    PROCESSED_PATH / "train.csv"
)

val_df = pd.read_csv(
    PROCESSED_PATH / "validation.csv"
)

test_df = pd.read_csv(
    PROCESSED_PATH / "test.csv"
)

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(3479, 12)
(435, 12)
(435, 12)


In [ ]:
#Convert to HF datasets

train_dataset = Dataset.from_pandas(
    train_df[["text", "label"]]
)

val_dataset = Dataset.from_pandas(
    val_df[["text", "label"]]
)

test_dataset = Dataset.from_pandas(
    test_df[["text", "label"]]
)

print(train_dataset)
print(val_dataset)
print(test_dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 3479
})
Dataset({
    features: ['text', 'label'],
    num_rows: 435
})
Dataset({
    features: ['text', 'label'],
    num_rows: 435
})


In [ ]:
#Load tokenizer + AfroXLMR

MODEL_NAME = "Davlan/afro-xlmr-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

print("Model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/398 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: Davlan/afro-xlmr-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded


In [ ]:
#Tokenize datasets

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

val_dataset = val_dataset.map(
    tokenize_function,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True
)

# Rename label to labels because Trainer expects "labels"
train_dataset = train_dataset.rename_column("label", "labels")
val_dataset = val_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")

# Remove raw text column so model only sees tensors
train_dataset = train_dataset.remove_columns(["text"])
val_dataset = val_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

# Set PyTorch format
train_dataset.set_format("torch")
val_dataset.set_format("torch")
test_dataset.set_format("torch")

print(train_dataset.column_names)
print(val_dataset.column_names)
print(test_dataset.column_names)

Map:   0%|          | 0/3479 [00:00<?, ? examples/s]

Map:   0%|          | 0/435 [00:00<?, ? examples/s]

Map:   0%|          | 0/435 [00:00<?, ? examples/s]

['labels', 'input_ids', 'attention_mask']
['labels', 'input_ids', 'attention_mask']
['labels', 'input_ids', 'attention_mask']


In [ ]:
#Training arguments


from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=str(MODEL_PATH / "afroxlmr_detector"),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none",
    save_total_limit=2
)

print("Training arguments ready")

Training arguments ready


In [ ]:
#Trainer setup

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro")
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print("Trainer ready")

Trainer ready


In [ ]:
#Train AfroXLMR

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,0.123852,0.972414,0.971264
2,0.249897,0.236367,0.949425,0.948120
3,0.109217,0.332866,0.944828,0.943568


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=1305, training_loss=0.1516352847161421, metrics={'train_runtime': 418.7105, 'train_samples_per_second': 24.927, 'train_steps_per_second': 3.117, 'total_flos': 686522521198080.0, 'train_loss': 0.1516352847161421, 'epoch': 3.0})

In [ ]:
#AfroXLMR test evaluation


predictions = trainer.predict(test_dataset)

y_pred = np.argmax(
    predictions.predictions,
    axis=1
)

y_true = predictions.label_ids

afro_accuracy = accuracy_score(
    y_true,
    y_pred
)

afro_macro_f1 = f1_score(
    y_true,
    y_pred,
    average="macro"
)

print("AfroXLMR Test Accuracy:", afro_accuracy)
print("AfroXLMR Test Macro F1:", afro_macro_f1)

print("\nClassification report:")
print(
    classification_report(
        y_true,
        y_pred
    )
)

print("\nConfusion matrix:")
print(
    confusion_matrix(
        y_true,
        y_pred
    )
)

AfroXLMR Test Accuracy: 0.9632183908045977
AfroXLMR Test Macro F1: 0.9618287118287119

Classification report:
              precision    recall  f1-score   support

           0       0.98      0.96      0.97       261
           1       0.94      0.97      0.95       174

    accuracy                           0.96       435
   macro avg       0.96      0.96      0.96       435
weighted avg       0.96      0.96      0.96       435


Confusion matrix:
[[251  10]
 [  6 168]]


In [ ]:
#Save AfroXLMR predictions


afro_results_df = test_df.copy()

afro_results_df["prediction"] = y_pred

afro_results_df["correct"] = (
    afro_results_df["label"] ==
    afro_results_df["prediction"]
)

afro_results_df.to_csv(
    RESULTS_PATH /
    "afroxlmr_test_predictions.csv",
    index=False
)

print("Saved AfroXLMR predictions")

Saved AfroXLMR predictions


In [ ]:
# Compare models

comparison_df = pd.DataFrame({
    "Model": [
        "Baseline TF-IDF + Logistic Regression",
        "AfroXLMR"
    ],
    "Accuracy": [
        baseline_test_accuracy,
        afro_accuracy
    ],
    "Macro_F1": [
        baseline_test_macro_f1,
        afro_macro_f1
    ]
})

display(comparison_df)

,Model,Accuracy,Macro_F1
0,Baseline TF-IDF + Logistic Regression,0.919540,0.916267
1,AfroXLMR,0.963218,0.961829


In [ ]:
#Error analysis

error_df = test_df.copy()

error_df["prediction"] = y_pred
error_df["correct"] = (
    error_df["label"] ==
    error_df["prediction"]
)

errors_only = error_df[
    error_df["correct"] == False
].copy()

errors_only.to_csv(
    RESULTS_PATH / "afroxlmr_errors.csv",
    index=False
)

print("Total mistakes:", len(errors_only))

print("\nFalse positives (human → machine):")
print(
    len(
        errors_only[
            (errors_only["label"] == 0) &
            (errors_only["prediction"] == 1)
        ]
    )
)

print("\nFalse negatives (machine → human):")
print(
    len(
        errors_only[
            (errors_only["label"] == 1) &
            (errors_only["prediction"] == 0)
        ]
    )
)

display(
    errors_only.sample(
        min(20, len(errors_only)),
        random_state=42
    )
)

Total mistakes: 16

False positives (human → machine):
10

False negatives (machine → human):
6


,text,language,language_code,source,label,model,temperature,generation_task,domain,length_style,seed_id,word_count,prediction,correct
40,Ziqhelanise nokukhangela abantwana bakho rhoqo...,xhosa,xho,vukuzenzele,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,False
63,Bašomomogolo ba mmušo ba hlaloša mabaka a go y...,sepedi,nso,groq,1,llama-3.3-70b-versatile,1.1,government_notice,clinic visits,one short sentence,2889.0,10.0,0,False
85,Re swanetše go ba le dipoledišano tša nnete e ...,sepedi,nso,vukuzenzele,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,False
371,Ditiro tše šoro tše ke go tshela ditokelo tša ...,sepedi,nso,vukuzenzele,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,False
306,Naa o latela melao ka moka ya COVID-19 yeo e ...,sepedi,nso,vukuzenzele,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,False
192,Mmusomogolo wa tša setšhaba o tlaleletše pono ...,sepedi,nso,groq,1,llama-3.3-70b-versatile,0.9,everyday_life,government policies,one short sentence,2569.0,21.0,0,False
161,Mmušo o sepetša gabotse ge a laetša seriti sa ...,sepedi,nso,groq,1,llama-3.3-70b-versatile,1.1,summarize,government policies,one short sentence,2557.0,10.0,0,False
172,Mopresidente Cyril Rama-phosa o tšebišitše gap...,sepedi,nso,groq,1,llama-3.3-70b-versatile,1.1,summarize,government policies,one short sentence,2471.0,13.0,0,False
66,Ditšhaba tša rena ga se tša swanela go dula le...,sepedi,nso,vukuzenzele,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,False
404,Batho ba bantši bao ba sa lwalego kudu ba tla ...,sepedi,nso,vukuzenzele,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,False


In [ ]:
#Error analysis by language


print("Errors by language:")
print(errors_only["language"].value_counts())

print("\nFalse positives by language:")
print(
    errors_only[
        (errors_only["label"] == 0) &
        (errors_only["prediction"] == 1)
    ]["language"].value_counts()
)

print("\nFalse negatives by language:")
print(
    errors_only[
        (errors_only["label"] == 1) &
        (errors_only["prediction"] == 0)
    ]["language"].value_counts()
)

Errors by language:
language
sepedi    11
xhosa      4
zulu       1
Name: count, dtype: int64

False positives by language:
language
sepedi    7
xhosa     3
Name: count, dtype: int64

False negatives by language:
language
sepedi    4
zulu      1
xhosa     1
Name: count, dtype: int64
